![logo_itq](img/logo-itq.jpeg)
## Selección de Atributos 2 - F-test e Información Mutua (Heart Disease)
*Nixon Malquin* — 24/05/2026

Usamos `f_classif` (ANOVA F-test) y `mutual_info_classif`. Para clasificación es el equivalente a `f_regression` / `mutual_info_regression` del Programa 6 original. Trabajamos sobre el dataset limpio (302 pacientes únicos).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import f_classif, mutual_info_classif, SelectKBest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('../dataset/heart_clean.csv')
X = df.drop(columns=['target'])
y = df['target'].values
cols = X.columns.tolist()
Xv = X.values
print('Pacientes unicos:', len(df))

## 1) F-test (`f_classif`)
Mide cuánto separa cada atributo a las clases (0=enfermo, 1=sano). Mayor F → más relevante; p-value > 0.05 sugiere irrelevancia.

In [ ]:
f_vals, f_pvals = f_classif(Xv, y)
order = np.argsort(f_vals)[::-1]
print(f"{'Rank':<6}{'Atributo':<12}{'F-score':>10}{'p-value':>12}")
for r, i in enumerate(order, 1):
    print(f'{r:<6}{cols[i]:<12}{f_vals[i]:>10.3f}{f_pvals[i]:>12.2e}')

plt.figure(figsize=(10,4))
plt.bar([cols[i] for i in order], f_vals[order])
plt.xticks(rotation=45); plt.ylabel('F'); plt.title('F-score por atributo')
plt.tight_layout(); plt.show()

## 2) Información Mutua (`mutual_info_classif`)
Captura dependencias lineales y no lineales con el target.

In [ ]:
mi = mutual_info_classif(Xv, y, random_state=42)
order_mi = np.argsort(mi)[::-1]
print(f"{'Rank':<6}{'Atributo':<12}{'MI':>10}")
for r, i in enumerate(order_mi, 1):
    print(f'{r:<6}{cols[i]:<12}{mi[i]:>10.4f}')

plt.figure(figsize=(10,4))
plt.bar([cols[i] for i in order_mi], mi[order_mi], color='orange')
plt.xticks(rotation=45); plt.ylabel('MI'); plt.title('Información mutua por atributo')
plt.tight_layout(); plt.show()

## 3) Barrido de K con Random Forest
Probamos cuántos atributos K dan el mejor balance de exactitud.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(Xv, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42, stratify=y_tr)

print(f"{'K':>3} {'val':>9} {'test':>9}  atributos")
for k in range(3, 14):
    sel = SelectKBest(score_func=f_classif, k=k)
    Xtr  = sel.fit_transform(X_train, y_train)
    Xval = sel.transform(X_val); Xte = sel.transform(X_te)
    rf = RandomForestClassifier(n_estimators=300, random_state=42)
    rf.fit(Xtr, y_train)
    va = rf.score(Xval, y_val); te = rf.score(Xte, y_te)
    elegidos = [cols[i] for i in sel.get_support(indices=True)]
    print(f'{k:>3} {va*100:>8.2f}% {te*100:>8.2f}%  {elegidos}')

### Conclusión
Las dos técnicas coinciden:
- `fbs` es irrelevante (p≈0.64, MI=0.007)
- `chol` y `restecg` son muy débiles
- Los más informativos son `exang, cp, oldpeak, thalach, ca, thal, slope, sex`

**K=8 maximiza la exactitud en test (~83.6%)**. Atributos finales:
`sex, cp, thalach, exang, oldpeak, slope, ca, thal`

### Link de repositorio
https://github.com/Ngmalquin123/MachingITQ